In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import pandas as pd
import numpy as np
from world_cup_2026.config import RAW_DATA_DIR
from world_cup_2026.data_ingestion.normalize import normalize_dataframe_teams

# Load all needed datasets
df_results   = pd.read_csv(RAW_DATA_DIR / "martj42_results" / "results.csv", parse_dates=["date"])
df_teams_2026 = pd.read_csv(RAW_DATA_DIR / "areezvisram12_fixture" / "teams.csv")

print(f"df_results   : {df_results.shape}")
print(f"df_teams_2026: {df_teams_2026.shape}")

df_results   : (49071, 9)
df_teams_2026: (48, 5)


In [4]:
# Normalize team names in df_results
df_results_norm = normalize_dataframe_teams(
    df_results,
    columns=["home_team", "away_team"]
)

# Verify USA fix
mask = (
    (df_results_norm["home_team"] == "USA") |
    (df_results_norm["away_team"] == "USA")
)
print(f"Matches with 'USA' after normalization: {mask.sum()}")
print(df_results_norm[mask][["date","home_team","away_team","tournament"]].tail(5).to_string())

2026-03-28 22:02:14.890 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Guernsey' — add to normalize.py if needed.
2026-03-28 22:02:14.898 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Jersey' — add to normalize.py if needed.
2026-03-28 22:02:14.900 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Jersey' — add to normalize.py if needed.
2026-03-28 22:02:14.906 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Guernsey' — add to normalize.py if needed.
2026-03-28 22:02:14.907 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Alderney' — add to normalize.py if needed.
2026-03-28 22:02:14.910 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Jersey' — add to normalize.py if needed.
2026-03-28

2026-03-28 22:02:14.974 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Basque Country' — add to normalize.py if needed.
2026-03-28 22:02:14.977 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Catalonia' — add to normalize.py if needed.
2026-03-28 22:02:14.980 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Basque Country' — add to normalize.py if needed.
2026-03-28 22:02:14.987 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Catalonia' — add to normalize.py if needed.
2026-03-28 22:02:15.006 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Catalonia' — add to normalize.py if needed.
2026-03-28 22:02:15.010 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Basque Country' — add to normali

Matches with 'USA' after normalization: 786
            date home_team  away_team tournament
48602 2025-09-09       USA      Japan   Friendly
48697 2025-10-10       USA    Ecuador   Friendly
48776 2025-10-14       USA  Australia   Friendly
48878 2025-11-15       USA   Paraguay   Friendly
48950 2025-11-18       USA    Uruguay   Friendly


In [15]:
from world_cup_2026.features.elo import calculate_elo

df_elo = calculate_elo(df_results_norm)
print(f"Elo calculated on normalized data: {df_elo.shape}")

2026-03-28 17:22:48.206 | INFO     | world_cup_2026.features.elo:calculate_elo:196 - Calculating Elo for 49,071 matches...
2026-03-28 17:22:50.851 | SUCCESS  | world_cup_2026.features.elo:calculate_elo:243 - Elo calculation complete. Registry has 333 teams.


Elo calculated on normalized data: (49071, 15)


In [16]:
confirmed_teams = df_teams_2026[df_teams_2026["is_placeholder"] == False]["team_name"].tolist()
all_teams_in_history = set(df_elo["home_team"].unique()) | set(df_elo["away_team"].unique())

print("WC2026 teams NOT found in historical results:")
missing = []
for team in confirmed_teams:
    if team not in all_teams_in_history:
        missing.append(team)
        print(f"  ✗ '{team}'")

print(f"\nTotal missing: {len(missing)} of {len(confirmed_teams)}")

WC2026 teams NOT found in historical results:
  ✗ 'Curaçao'
  ✗ 'IR Iran'
  ✗ 'Cabo Verde'

Total missing: 3 of 42


In [17]:
# Verify current Elo for WC2026 top teams
from world_cup_2026.features.elo import get_elo_at_date

teams = ["Argentina", "France", "Brazil", "England", "Spain", "Germany"]
cutoff = pd.Timestamp("2026-01-01")

print("Current Elo ratings (as of Jan 2026):")
for team in teams:
    elo = get_elo_at_date(df_elo, team, cutoff)
    print(f"  {team:<15} {elo:.1f}")

Current Elo ratings (as of Jan 2026):
  Argentina       2153.1
  France          2106.2
  Brazil          2015.6
  England         2089.6
  Spain           2195.2
  Germany         2006.4


In [18]:
from world_cup_2026.features.h2h import H2HAnalyzer

# Initialize analyzer with Elo-enriched data
analyzer = H2HAnalyzer(df_elo, window_months=48)

# Test 1 — Argentina vs Brazil (classic H2H)
cutoff = pd.Timestamp("2026-06-01")
features_arg_bra = analyzer.get_matchup_features("Argentina", "Brazil", as_of=cutoff)
print("Argentina vs Brazil — H2H features:")
for k, v in features_arg_bra.items():
    if k not in ["team_a", "team_b", "as_of"]:
        print(f"  {k:<35} {v}")

2026-03-28 17:24:24.686 | INFO     | world_cup_2026.features.h2h:__init__:70 - H2HAnalyzer initialized — 49,071 matches, window=48m, decay_lambda=0.02


Argentina vs Brazil — H2H features:
  h2h_matches                         2
  h2h_win_rate_a                      1.0
  h2h_goal_diff_a                     2.1595962877213286
  h2h_elo_edge_a                      0.36684783937216203
  h2h_weighted_edge_a                 0.23802629529993807
  h2h_decay_weight                    0.648842025912721
  h2h_reliable                        True
  transitive_common_rivals            11
  transitive_edge_a                   0.05539537453982081
  transitive_goal_diff_edge           0.6116967984535185
  transitive_reliable                 True


In [19]:
# Test 2 — WC2026 Group stage matchups
matchups = [
    ("Argentina", "France",  pd.Timestamp("2026-06-01")),
    ("Brazil",    "Spain",   pd.Timestamp("2026-06-01")),
    ("England",   "Germany", pd.Timestamp("2026-06-01")),
    ("Mexico",    "USA",     pd.Timestamp("2026-06-01")),
]

df_h2h = analyzer.build_feature_matrix(matchups)
print(df_h2h[["team_a", "team_b", "h2h_matches", "h2h_elo_edge_a",
               "transitive_common_rivals", "transitive_edge_a"]].to_string(index=False))

2026-03-28 17:24:29.129 | INFO     | world_cup_2026.features.h2h:build_feature_matrix:436 - Building H2H feature matrix for 4 matchups...
2026-03-28 17:24:29.214 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Argentina vs France: only 1 matches (min=2) — returning neutral features.
2026-03-28 17:24:29.568 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Brazil vs Spain: only 1 matches (min=2) — returning neutral features.
2026-03-28 17:24:30.596 | SUCCESS  | world_cup_2026.features.h2h:build_feature_matrix:442 - H2H feature matrix built: (4, 14)


   team_a  team_b  h2h_matches  h2h_elo_edge_a  transitive_common_rivals  transitive_edge_a
Argentina  France            1         0.00000                         7           0.230980
   Brazil   Spain            1         0.00000                         8           0.036122
  England Germany            2        -0.04606                        14          -0.160949
   Mexico     USA            5         0.06955                        23          -0.016316


In [20]:
# Diagnose Mexico vs USA — find their actual last matches
mask_mex_usa = (
    ((df_elo["home_team"] == "Mexico") & (df_elo["away_team"] == "United States"))
    | ((df_elo["home_team"] == "United States") & (df_elo["away_team"] == "Mexico"))
)
print("Mexico vs USA recent matches:")
print(df_elo[mask_mex_usa][["date","home_team","away_team","home_score","away_score","tournament"]].tail(10).to_string(index=False))

Mexico vs USA recent matches:
Empty DataFrame
Columns: [date, home_team, away_team, home_score, away_score, tournament]
Index: []


In [21]:
# Also check with "USA" vs normalize
mask_mex_usa2 = (
    ((df_elo["home_team"] == "Mexico") & (df_elo["away_team"] == "USA"))
    | ((df_elo["home_team"] == "USA") & (df_elo["away_team"] == "Mexico"))
)
print("\nMexico vs USA (alternate name):")
print(df_elo[mask_mex_usa2][["date","home_team","away_team","home_score","away_score","tournament"]].tail(10).to_string(index=False))


Mexico vs USA (alternate name):
      date home_team away_team  home_score  away_score                   tournament
2019-09-06       USA    Mexico           0           3                     Friendly
2021-06-06       USA    Mexico           3           2      CONCACAF Nations League
2021-08-01       USA    Mexico           1           0                     Gold Cup
2021-11-12       USA    Mexico           2           0 FIFA World Cup qualification
2022-03-24    Mexico       USA           0           0 FIFA World Cup qualification
2023-04-19       USA    Mexico           1           1                     Friendly
2023-06-15       USA    Mexico           3           0      CONCACAF Nations League
2024-03-24       USA    Mexico           2           0      CONCACAF Nations League
2024-10-15    Mexico       USA           2           0                     Friendly
2025-07-06       USA    Mexico           1           2                     Gold Cup


In [24]:
from world_cup_2026.data_ingestion.normalize import normalize_team_name

# Normalize fixture team names to match historical data
df_teams_2026["team_name_norm"] = df_teams_2026["team_name"].map(normalize_team_name)

# Re-run diagnosis with normalized names
confirmed_teams_norm = df_teams_2026[
    df_teams_2026["is_placeholder"] == False
]["team_name_norm"].tolist()

print("WC2026 teams NOT found after normalizing fixture:")
missing = []
for team in confirmed_teams_norm:
    if team not in all_teams_in_history:
        missing.append(team)
        print(f"  ✗ '{team}'")

print(f"\nTotal missing: {len(missing)} of {len(confirmed_teams_norm)}")
print(f"\nNormalized fixture names sample:")
print(df_teams_2026[["team_name","team_name_norm"]].to_string(index=False))

2026-03-28 17:26:35.700 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:186 - Unknown team name: 'Winner UEFA Playoff D' — add to normalize.py if needed.
2026-03-28 17:26:35.703 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:186 - Unknown team name: 'Winner UEFA Playoff A' — add to normalize.py if needed.
2026-03-28 17:26:35.705 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:186 - Unknown team name: 'Winner UEFA Playoff C' — add to normalize.py if needed.
2026-03-28 17:26:35.707 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:186 - Unknown team name: 'Winner UEFA Playoff B' — add to normalize.py if needed.
2026-03-28 17:26:35.711 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:186 - Unknown team name: 'Winner FIFA Playoff 2' — add to normalize.py if needed.
2026-03-28 17:26:35.714 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:186 - Un

WC2026 teams NOT found after normalizing fixture:
  ✗ 'Cabo Verde'

Total missing: 1 of 42

Normalized fixture names sample:
            team_name        team_name_norm
               Mexico                Mexico
         South Africa          South Africa
          South Korea           South Korea
Winner UEFA Playoff D Winner UEFA Playoff D
               Canada                Canada
Winner UEFA Playoff A Winner UEFA Playoff A
                Qatar                 Qatar
          Switzerland           Switzerland
               Brazil                Brazil
              Morocco               Morocco
                Haiti                 Haiti
             Scotland              Scotland
                  USA                   USA
             Paraguay              Paraguay
            Australia             Australia
Winner UEFA Playoff C Winner UEFA Playoff C
              Germany               Germany
              Curaçao               Curacao
        Côte d'Ivoire         Côte d'Iv

In [25]:
import importlib
import world_cup_2026.data_ingestion.normalize as norm_module
importlib.reload(norm_module)
from world_cup_2026.data_ingestion.normalize import normalize_team_name

# Re-normalize fixture
df_teams_2026["team_name_norm"] = df_teams_2026["team_name"].map(normalize_team_name)

confirmed_teams_norm = df_teams_2026[
    df_teams_2026["is_placeholder"] == False
]["team_name_norm"].tolist()

missing = [t for t in confirmed_teams_norm if t not in all_teams_in_history]
print(f"Missing after fix: {missing if missing else 'None ✓'}")
print(f"Matched: {len(confirmed_teams_norm) - len(missing)}/42")

2026-03-28 17:29:07.796 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Winner UEFA Playoff D' — add to normalize.py if needed.
2026-03-28 17:29:07.812 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Winner UEFA Playoff A' — add to normalize.py if needed.
2026-03-28 17:29:07.813 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Winner UEFA Playoff C' — add to normalize.py if needed.
2026-03-28 17:29:07.816 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Winner UEFA Playoff B' — add to normalize.py if needed.
2026-03-28 17:29:07.820 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Unknown team name: 'Winner FIFA Playoff 2' — add to normalize.py if needed.
2026-03-28 17:29:07.822 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:189 - Un

Missing after fix: None ✓
Matched: 42/42


In [5]:
from world_cup_2026.features.form import FormCalculator

calc = FormCalculator(df_results_norm, windows=[5, 10, 20])

# Test current form for WC2026 top teams
teams_to_check = ["Argentina", "France", "Brazil", "Spain", "England", "Germany", "USA", "Mexico"]
cutoff = pd.Timestamp("2026-06-01")

print(f"{'Team':<15} {'W10':>6} {'Pts10':>7} {'GF10':>6} {'GA10':>6} {'WPts10':>8}")
print("-" * 55)
for team in teams_to_check:
    f = calc.get_team_current_form(team, as_of=cutoff, window=10)
    print(
        f"{team:<15} "
        f"{f['form_10_win_rate']:>6.3f} "
        f"{f['form_10_points_avg']:>7.3f} "
        f"{f['form_10_goals_scored_avg']:>6.3f} "
        f"{f['form_10_goals_conceded_avg']:>6.3f} "
        f"{f['form_10_weighted_points']:>8.3f}"
    )

2026-03-28 22:03:25.237 | INFO     | world_cup_2026.features.form:__init__:81 - FormCalculator initialized — 49,071 matches, windows=[5, 10, 20], decay_lambda=0.1


Team               W10   Pts10   GF10   GA10   WPts10
-------------------------------------------------------
Argentina        0.800   2.500  2.000  0.300    2.483
France           0.700   2.200  2.400  1.100    2.346
Brazil           0.500   1.700  1.700  1.000    1.674
Spain            0.600   2.200  3.300  1.300    2.276
England          0.900   2.700  2.600  0.300    2.752
Germany          0.600   1.900  2.200  1.100    2.119
USA              0.600   2.000  1.900  1.200    2.120
Mexico           0.400   1.600  0.900  1.000    1.617


In [6]:
# Test on a small subset first (full computation takes a few minutes on 49k rows)
df_recent = df_results_norm[df_results_norm["date"].dt.year >= 2020].copy()
print(f"Computing form on recent subset: {len(df_recent):,} matches")

df_form = calc.compute_form_features(df_recent)
print(f"\nShape after form features: {df_form.shape}")
print(f"New columns added: {len(df_form.columns) - len(df_recent.columns)}")

# Sample output
form_cols = [c for c in df_form.columns if "form_10_win" in c or "form_10_points" in c]
print(f"\nSample form columns: {form_cols}")
print(df_form[["date","home_team","away_team"] + form_cols].tail(5).to_string(index=False))

Computing form on recent subset: 5,732 matches


2026-03-28 22:06:22.262 | INFO     | world_cup_2026.features.form:compute_form_features:241 - Computing form features (all matches) for 5,732 rows...
2026-03-28 22:07:02.242 | SUCCESS  | world_cup_2026.features.form:compute_form_features:298 - Form features computed. Added 66 columns.



Shape after form features: (5732, 75)
New columns added: 66

Sample form columns: ['home_form_10_win_rate', 'home_form_10_points_avg', 'away_form_10_win_rate', 'away_form_10_points_avg']
      date  home_team away_team  home_form_10_win_rate  home_form_10_points_avg  away_form_10_win_rate  away_form_10_points_avg
2026-01-18    Grenada   Jamaica                    0.6                      1.9                    0.5                      1.7
2026-01-18    Morocco   Senegal                    0.8                      2.6                    0.8                      2.5
2026-01-22     Panama    Mexico                    0.5                      2.0                    0.3                      1.4
2026-01-25    Bolivia    Mexico                    0.3                      1.1                    0.4                      1.6
2026-01-26 Uzbekistan     China                    0.5                      1.9                    0.3                      0.9
